# UBridge AI Guide: PyTorch semantic search
Run the cells top to bottom. A CPU runtime is enough; a GPU only makes training faster.

## 1. Get the project

In [ ]:
REPO_URL = "https://github.com/erich-gagui18/UBridge"
BRANCH = "main"

In [ ]:
import glob, os, subprocess
if REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "UBridge"], check=True)
else:
    from google.colab import files
    uploaded = files.upload()
    subprocess.run(["unzip", "-o", "-q", next(iter(uploaded))], check=True)
ml = os.path.dirname(glob.glob("**/ml/evaluate.py", recursive=True)[0])
os.chdir(ml)
print(os.getcwd())

c:\4th Year 1st Sem\CAP102\UBridge\ml\colab\UBridge\ml


In [3]:
!pip -q install -r requirements-train.txt pytest


[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Smoke test
Trains a tiny random model for two epochs. It must pass before you spend time on the real run.

In [4]:
!python -m pytest tests/test_train_smoke.py -q

.                                                                        [100%]
============================== warnings summary ===============================
tests/test_train_smoke.py::test_one_training_run_produces_a_saved_model_and_a_finite_loss
  c:\4th Year 1st Sem\CAP102\UBridge\ml\colab\UBridge\ml\tests\test_train_smoke.py:14: DeprecationWarning: Importing from 'sentence_transformers.models' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.modules' instead.
    from sentence_transformers import SentenceTransformer, models

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
1 passed, 1 warning in 43.32s


## 3. Baseline: the keyword search that is live now

In [5]:
!python evaluate.py

# Retrieval evaluation (test split, handbook saep-m-rev4-2026)

hit@k: a correct chunk is in the top k. MRR@4: mean reciprocal rank of the first correct chunk. No result: in-scope questions with nothing returned. Rejection: out-of-scope questions correctly answered with nothing.

| System | In-scope questions | hit@1 | hit@4 | MRR@4 | No result | Out-of-scope rejected |
|---|---|---|---|---|---|---|
| Keyword (BM25, current) | 41 | 58.5% | 68.3% | 0.628 | 24.4% | 72.7% |

## hit@4 by question language (en = English, tl = Filipino/Taglish)

| System | en | tl |
|---|---|---|
| Keyword (BM25, current) | 82.4% | 0.0% |

Thresholds are calibrated on the dev split (18 questions) and reported on the test split.



## 4. Pretrained embedding models, no training

In [6]:
!python evaluate.py --models intfloat/multilingual-e5-small intfloat/multilingual-e5-base sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

# Retrieval evaluation (test split, handbook saep-m-rev4-2026)

hit@k: a correct chunk is in the top k. MRR@4: mean reciprocal rank of the first correct chunk. No result: in-scope questions with nothing returned. Rejection: out-of-scope questions correctly answered with nothing.

| System | In-scope questions | hit@1 | hit@4 | MRR@4 | No result | Out-of-scope rejected |
|---|---|---|---|---|---|---|
| Keyword (BM25, current) | 41 | 58.5% | 68.3% | 0.628 | 24.4% | 72.7% |
| Dense: intfloat/multilingual-e5-small (threshold 0.8072) | 41 | 61.0% | 78.0% | 0.683 | 7.3% | 45.5% |
| Hybrid (BM25 + dense): intfloat/multilingual-e5-small | 41 | 63.4% | 80.5% | 0.707 | 7.3% | 45.5% |
| Dense: intfloat/multilingual-e5-base (threshold 0.795) | 41 | 63.4% | 75.6% | 0.689 | 9.8% | 54.5% |
| Hybrid (BM25 + dense): intfloat/multilingual-e5-base | 41 | 65.9% | 82.9% | 0.732 | 7.3% | 54.5% |
| Dense: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 (threshold 0.5481) | 41 | 39.0% | 39.0% | 0.

C:\Users\erich\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\erich\.cache\huggingface\hub\models--intfloat--multilingual-e5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\erich\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py

## 5. Fine-tune with PyTorch (optional)
Pick the model that scored best above.

In [7]:
MODEL = "intfloat/multilingual-e5-small"

In [8]:
!python make_pairs.py
!python train.py --model $MODEL --out models/ubridge-embedder --epochs 4

302 training pairs written to C:\4th Year 1st Sem\CAP102\UBridge\ml\colab\UBridge\ml\eval\train_pairs.jsonl
epoch 0 (pretrained): dev hit@4 = 0.909
epoch 1: loss = 0.9246501010927287, dev hit@4 = 0.727
epoch 2: loss = 0.2717521238130179, dev hit@4 = 0.818
epoch 3: loss = 0.10007705063564407, dev hit@4 = 0.727
epoch 4: loss = 0.05131281896647124, dev hit@4 = 0.727



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1734.24it/s]

Writing model shards: 100%|██████████| 1/1 [00:30<00:00, 30.16s/it]
The `tokenize` method is deprecated, please use `preprocess` instead.


In [9]:
!python evaluate.py --models $MODEL models/ubridge-embedder

# Retrieval evaluation (test split, handbook saep-m-rev4-2026)

hit@k: a correct chunk is in the top k. MRR@4: mean reciprocal rank of the first correct chunk. No result: in-scope questions with nothing returned. Rejection: out-of-scope questions correctly answered with nothing.

| System | In-scope questions | hit@1 | hit@4 | MRR@4 | No result | Out-of-scope rejected |
|---|---|---|---|---|---|---|
| Keyword (BM25, current) | 41 | 58.5% | 68.3% | 0.628 | 24.4% | 72.7% |
| Dense: intfloat/multilingual-e5-small (threshold 0.8072) | 41 | 61.0% | 78.0% | 0.683 | 7.3% | 45.5% |
| Hybrid (BM25 + dense): intfloat/multilingual-e5-small | 41 | 63.4% | 80.5% | 0.707 | 7.3% | 45.5% |
| Dense: models/ubridge-embedder (threshold 0.8193) | 41 | 61.0% | 82.9% | 0.697 | 4.9% | 45.5% |
| Hybrid (BM25 + dense): models/ubridge-embedder | 41 | 65.9% | 85.4% | 0.738 | 4.9% | 45.5% |

## hit@4 by question language (en = English, tl = Filipino/Taglish)

| System | en | tl |
|---|---|---|
| Keyword (BM25, cu


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1492.72it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1574.90it/s]


## 6. Build the search index
Set BEST to the winner. A fine-tuned model must be published to the Hugging Face Hub (set HUB_ID) so the service can download it.

In [10]:
BEST = "intfloat/multilingual-e5-small"
HUB_ID = ""

In [11]:
if BEST.startswith("models/") and not HUB_ID:
    raise SystemExit("Set HUB_ID (for example your-username/ubridge-embedder) to publish the fine-tuned model.")
if HUB_ID:
    from huggingface_hub import notebook_login
    from sentence_transformers import SentenceTransformer
    notebook_login()
    SentenceTransformer(BEST).push_to_hub(HUB_ID)
record = f"--record-as {HUB_ID}" if HUB_ID else ""
!python build_index.py --model $BEST $record

{
  "model": "intfloat/multilingual-e5-small",
  "handbook_version": "saep-m-rev4-2026",
  "handbook_hash": "edf7fe4d942227e2f2ffcb5606e957211cf5d7cfcb8fda13b37c352ce4988741",
  "dimension": 384,
  "chunks": 34,
  "query_prefix": "query: ",
  "passage_prefix": "passage: ",
  "min_score": 0.8072,
  "built_at": "2026-10-21T09:20:38+00:00"
}



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1787.53it/s]


In [12]:
!python evaluate.py --models $BEST

# Retrieval evaluation (test split, handbook saep-m-rev4-2026)

hit@k: a correct chunk is in the top k. MRR@4: mean reciprocal rank of the first correct chunk. No result: in-scope questions with nothing returned. Rejection: out-of-scope questions correctly answered with nothing.

| System | In-scope questions | hit@1 | hit@4 | MRR@4 | No result | Out-of-scope rejected |
|---|---|---|---|---|---|---|
| Keyword (BM25, current) | 41 | 58.5% | 68.3% | 0.628 | 24.4% | 72.7% |
| Dense: intfloat/multilingual-e5-small (threshold 0.8072) | 41 | 61.0% | 78.0% | 0.683 | 7.3% | 45.5% |
| Hybrid (BM25 + dense): intfloat/multilingual-e5-small | 41 | 63.4% | 80.5% | 0.707 | 7.3% | 45.5% |

## hit@4 by question language (en = English, tl = Filipino/Taglish)

| System | en | tl |
|---|---|---|
| Keyword (BM25, current) | 82.4% | 0.0% |
| Dense: intfloat/multilingual-e5-small | 88.2% | 28.6% |
| Hybrid: intfloat/multilingual-e5-small | 91.2% | 28.6% |

Thresholds are calibrated on the dev split (18 ques


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1692.86it/s]


## 7. Download the files to commit

In [13]:
import shutil
shutil.make_archive("ubridge-ml-artifacts", "zip", "artifacts")
from google.colab import files
files.download("ubridge-ml-artifacts.zip")

ModuleNotFoundError: No module named 'google'